# Segmentacja Obrazu: Binaryzacja Globalna i Adaptacyjna

Niniejszy notatnik skupia się na technikach segmentacji obrazu, których celem jest wyodrębnienie obiektów pierwszoplanowych od tła (Region of Interest - ROI). 

Projekt demonstruje ewolucję podejść do binaryzacji – od prostego progowania na podstawie analizy histogramu, poprzez algorytmy automatycznej optymalizacji progu (metoda iteracyjna oraz metoda Otsu), aż po zaawansowane metody lokalne (Sauvola), które skutecznie radzą sobie z degradacją obrazu spowodowaną szumem oraz niejednorodnym oświetleniem sceny.

In [ ]:
import matplotlib.pyplot as plt
import cv2
import numpy as np
import os

if not os.path.exists("coins.png") :
    !wget https://raw.githubusercontent.com/vision-agh/poc_sw/master/04_Thresholding/coins.png --no-check-certificate
if not os.path.exists("rice.png") :
    !wget https://raw.githubusercontent.com/vision-agh/poc_sw/master/04_Thresholding/rice.png --no-check-certificate
if not os.path.exists("catalogue.png") :
    !wget https://raw.githubusercontent.com/vision-agh/poc_sw/master/04_Thresholding/catalogue.png --no-check-certificate
if not os.path.exists("bart.png") :
    !wget https://raw.githubusercontent.com/vision-agh/poc_sw/master/04_Thresholding/bart.png --no-check-certificate
if not os.path.exists("figura1.png") :
    !wget https://raw.githubusercontent.com/vision-agh/poc_sw/master/04_Thresholding/figura1.png --no-check-certificate
if not os.path.exists("figura2.png") :
    !wget https://raw.githubusercontent.com/vision-agh/poc_sw/master/04_Thresholding/figura2.png --no-check-certificate
if not os.path.exists("figura3.png") :
    !wget https://raw.githubusercontent.com/vision-agh/poc_sw/master/04_Thresholding/figura3.png --no-check-certificate
if not os.path.exists("figura4.png") :
    !wget https://raw.githubusercontent.com/vision-agh/poc_sw/master/04_Thresholding/figura4.png --no-check-certificate

coins = cv2.imread("coins.png", cv2.IMREAD_GRAYSCALE)
plt.imshow(coins, cmap="gray")
plt.axis("off")
plt.show()

coins_hist = np.histogram(coins, bins=256, range=(0, 255))
plt.Figure(figsize=(10, 5))
plt.plot(coins_hist[1][:-1], coins_hist[0])
plt.xticks(np.arange(0, 256, 20))
plt.rcParams["figure.figsize"] = (10, 5)
plt.title("Histogram of coins")
plt.show()

figura1 = cv2.imread("figura1.png", cv2.IMREAD_GRAYSCALE)
plt.imshow(figura1, cmap="gray")
plt.axis("off")
plt.show()

figura1_hist = np.histogram(figura1, bins=256, range=(0, 255))
plt.Figure(figsize=(10, 5))
plt.plot(figura1_hist[1][:-1], figura1_hist[0])
plt.xticks(np.arange(0, 256, 20))
plt.rcParams["figure.figsize"] = (10, 5)
plt.title("Histogram of figura1")
plt.show()  

figura2 = cv2.imread("figura2.png", cv2.IMREAD_GRAYSCALE)
plt.imshow(figura2, cmap="gray")
plt.axis("off")
plt.show()

figura2_hist = np.histogram(figura2, bins=256, range=(0, 255))
plt.Figure(figsize=(10, 5))
plt.plot(figura2_hist[1][:-1], figura2_hist[0])
plt.xticks(np.arange(0, 256, 20))
plt.rcParams["figure.figsize"] = (10, 5)
plt.title("Histogram of figura2")
plt.show()

figura3 = cv2.imread("figura3.png", cv2.IMREAD_GRAYSCALE)
plt.imshow(figura3, cmap="gray")
plt.axis("off")
plt.show()      
figura3_hist = np.histogram(figura3, bins=256, range=(0, 255))
plt.Figure(figsize=(10, 5))
plt.plot(figura3_hist[1][:-1], figura3_hist[0])
plt.xticks(np.arange(0, 256, 20))
plt.rcParams["figure.figsize"] = (10, 5)
plt.title("Histogram of figura3")
plt.show()

figura4 = cv2.imread("figura4.png", cv2.IMREAD_GRAYSCALE)
plt.imshow(figura4, cmap="gray")
plt.axis("off")
plt.show()
figura4_hist = np.histogram(figura4, bins=256, range=(0, 255))
plt.Figure(figsize=(10, 5))
plt.plot(figura4_hist[1][:-1], figura4_hist[0])
plt.xticks(np.arange(0, 256, 20))
plt.rcParams["figure.figsize"] = (10, 5)
plt.title("Histogram of figura4")
plt.show()  


## 1. Binaryzacja Globalna (Ręczna analiza bimodalna)
W idealnych warunkach oświetleniowych, histogram obrazu składa się z dwóch wyraźnych maksimów (bimodalność) - jedno odpowiada obiektom, drugie tłu. Wyznaczenie punktu odcięcia (progu) pomiędzy tymi maksimami pozwala na błyskawiczną segmentację.

In [ ]:
def binaryzacja(img, threshold):
    result = np.zeros(img.shape, dtype=np.uint8)
    result[img > threshold] = 255
    plt.imshow(result, cmap="gray")
    plt.axis("off")
    plt.show()
    return result


binary_coins = binaryzacja(coins, 79)

### Problem Szumu i Gradientu Oświetlenia
Poniższe testy obrazują wpływ zakłóceń środowiskowych na sztywny próg binaryzacji. Wprowadzenie szumu Gaussowskiego lub nieliniowego oświetlenia (np. winietowania obiektywu) powoduje zatarcie granicy między tłem a obiektem.

In [ ]:
binary_figura1 = binaryzacja(figura1, 79)
binary_figura2 = binaryzacja(figura2, 110)
binary_figura3 = binaryzacja(figura3, 170)
binary_figura4 = binaryzacja(figura4, 46)


In [ ]:
"Nie jest mozliwe we wszystkich przypadkach, poniewaz szum moze powodowac, ze piksele tła beda mialy jasnosc wieksza niz prog, co spowoduje, ze beda traktowane jako obiekty. Dodatkowo, jesli obiekty maja podobna jasnosc do tla, to moga byc niewidoczne po binaryzacji."

## 2. Automatyczna Optymalizacja Progu

W środowiskach przemysłowych manualne dobieranie progu jest nieakceptowalne. Poniżej zaimplementowano dwie metody automatycznej estymacji punktu odcięcia:

**1. Algorytm Iteracyjny:** Wykorzystuje średnią jasność obrazu jako punkt startowy, dzieląc piksele na dwie klasy. Następnie wyznacza nowe średnie dla obu klas i aktualizuje próg do momentu stabilizacji błędu.

**2. Metoda Otsu (Maximum Variance):** Algorytm wyczerpujący (optymalny). Przeszukuje całą przestrzeń histogramu w poszukiwaniu progu $k$, który zmaksymalizuje wariancję międzyklasową $\sigma^2_B(k)$, minimalizując tym samym błąd przypisania pikseli na granicy klas.

In [ ]:
def iterative_thresholding(img):
    hist = cv2.calcHist([img], [0], None, [256], [0, 256])
    hist = np.squeeze(hist)
    
    N = img.shape[0] * img.shape[1]
    p_i = hist / N
    
    P_0 = np.cumsum(p_i)
    i_vals = np.arange(256)
    
    k = int(np.dot(i_vals, p_i))

    while True:
        P0_k = P_0[k]
        P1_k = 1.0 - P0_k

        if P0_k == 0 or P1_k == 0:
            break
        m0 = (1 / P0_k) * np.dot(i_vals[:k+1], p_i[:k+1])

        m1 = (1 / P1_k) * np.dot(i_vals[k+1:], p_i[k+1:])
        
        k_new = int((m0 + m1) / 2)

        if abs(k_new - k) < 1:
            break
            
        k = k_new

    _, binarized = cv2.threshold(img, k, 255, cv2.THRESH_BINARY)

    plt.figure(figsize=(12, 5))
    
    plt.subplot(1, 2, 1)
    plt.imshow(img, cmap='gray', vmin=0, vmax=255)
    plt.axis('off')
    
    plt.subplot(1, 2, 2)
    plt.imshow(binarized, cmap='gray', vmin=0, vmax=255)
    plt.title(f"Binaryzacja (Wyznaczony próg: {k})")
    plt.axis('off')
    plt.show()

    return k

images_to_test = [
coins, figura1, figura2, figura3, figura4
]

for img_name in images_to_test:
    iterative_thresholding(img_name)


In [ ]:
def otsu_thresholding(img):
    hist = cv2.calcHist([img], [0], None, [256], [0, 256])
    hist = np.squeeze(hist)
    
    N = img.shape[0] * img.shape[1]
    p_i = hist / N
    
    i_vals = np.arange(256)
    m_G = np.dot(i_vals, p_i)

    sigma_b_squared = np.zeros(256)
    P_0 = 0.0
    m_k = 0.0

    for k in range(256):
        P_0 += p_i[k]
        m_k += k * p_i[k]
        
        if 0 < P_0 < 1:
            sigma_b_squared[k] = ((m_G * P_0 - m_k) ** 2) / (P_0 * (1.0 - P_0))

    k_bar = int(np.argmax(sigma_b_squared))

    _, binarized = cv2.threshold(img, k_bar, 255, cv2.THRESH_BINARY)
    k_cv2, binarized_cv2 = cv2.threshold(img, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    plt.figure(figsize=(16, 4))
    
    plt.subplot(1, 4, 1)
    plt.imshow(img, cmap='gray', vmin=0, vmax=255)
    plt.axis('off')
    
    plt.subplot(1, 4, 2)
    plt.plot(sigma_b_squared)
    plt.title("Wariancja")
    
    plt.subplot(1, 4, 3)
    plt.imshow(binarized, cmap='gray', vmin=0, vmax=255)
    plt.title(f"Otsu (wlasne): {k_bar}")
    plt.axis('off')

    plt.subplot(1, 4, 4)
    plt.imshow(binarized_cv2, cmap='gray', vmin=0, vmax=255)
    plt.title(f"Otsu (CV2): {int(k_cv2)}")
    plt.axis('off')
    plt.show()

    return k_bar

rice = cv2.imread("rice.png", cv2.IMREAD_GRAYSCALE)
catalogue = cv2.imread("catalogue.png", cv2.IMREAD_GRAYSCALE)

images_to_test = [
    coins, figura1, figura2, figura3, figura4, rice, catalogue
]

for img_name in images_to_test:
    otsu_thresholding(img_name)

## 3. Kompensacja Gradientu Oświetlenia (Binaryzacja Lokalna)

Wydajność algorytmów globalnych dramatycznie spada w przypadku oświetlenia kierunkowego lub winietowania (widoczne na zdjęciu `rice.png`). Rozwiązaniem jest podejście adaptacyjne (lokalne), w którym próg odcięcia kalkulowany jest dynamicznie dla każdego piksela na podstawie statystyki jego bezpośredniego otoczenia (okna analizy).

In [ ]:
def local_binarization(img, W=15):
    X, Y = img.shape
    binarized = np.zeros((X, Y), dtype=np.uint8)
    
    half_W = int(W / 2)

    for i in range(half_W, X - half_W):
        for j in range(half_W, Y - half_W):
            window = img[i - half_W : i + half_W + 1, j - half_W : j + half_W + 1]
            threshold = np.mean(window)
            
            if img[i, j] >= threshold:
                binarized[i, j] = 255
            else:
                binarized[i, j] = 0

    plt.figure(figsize=(10, 5))
    
    plt.subplot(1, 2, 1)
    plt.imshow(img, cmap='gray', vmin=0, vmax=255)
    plt.title("Oryginal")
    plt.axis('off')
    
    plt.subplot(1, 2, 2)
    plt.imshow(binarized, cmap='gray', vmin=0, vmax=255)
    plt.title(f"Lokalna (W={W})")
    plt.axis('off')
    plt.show()

images_to_test = [rice, catalogue]

for img in images_to_test:
    local_binarization(img, 15)

### Algorytm Sauvola-Pietikäinen
Zwykła średnia ruchoma często zawodzi w miejscach pustych (np. czyste tło tekstowe), generując fałszywe obiekty. Algorytm Sauvoli rozwiązuje ten problem poprzez analizę wariancji (odchylenia standardowego) w oknie. W obszarach gładkich próg zostaje sztucznie obniżony, co skutecznie wygładza tło, zachowując jednocześnie ostre krawędzie czcionki.

In [ ]:
def sauvola_binarization(img, W=15, k=0.15, R=128, sign=1):
    X, Y = img.shape
    binarized = np.zeros((X, Y), dtype=np.uint8)
    
    half_W = int(W / 2)

    for i in range(half_W, X - half_W):
        for j in range(half_W, Y - half_W):
            window = img[i - half_W : i + half_W + 1, j - half_W : j + half_W + 1]
            m = np.mean(window)
            s = np.std(window)
            
            threshold = m * (1 + sign * k * (s / R - 1))
            
            if img[i, j] >= threshold:
                binarized[i, j] = 255
            else:
                binarized[i, j] = 0

    plt.figure(figsize=(10, 5))
    
    plt.subplot(1, 2, 1)
    plt.imshow(img, cmap='gray', vmin=0, vmax=255)
    plt.title("Oryginal")
    plt.axis('off')
    
    plt.subplot(1, 2, 2)
    plt.imshow(binarized, cmap='gray', vmin=0, vmax=255)
    sign_str = "+" if sign == 1 else "-"
    plt.title(f"Sauvola (W={W}, k={k}, {sign_str})")
    plt.axis('off')
    plt.show()

sauvola_binarization(rice, W=15, k=0.15, R=128, sign=-1)
sauvola_binarization(rice, W=15, k=0.15, R=128, sign=1)
sauvola_binarization(catalogue, W=15, k=0.15, R=128, sign=1)
sauvola_binarization(catalogue, W=15, k=0.15, R=128, sign=-1)

## 4. Segmentacja Przedziałowa (Binaryzacja Dwuprogowa)
W przeciwieństwie do standardowego odcinania "powyżej/poniżej", metoda dwuprogowa definiuje "okno" akceptowalnej jasności. Pozwala to na selektywną izolację pikseli o specyficznej charakterystyce, co w połączeniu z logiką AND jest potężnym narzędziem np. do detekcji koloru skóry.

In [ ]:
bart = cv2.imread('bart.png', cv2.IMREAD_GRAYSCALE)

hist = cv2.calcHist([bart], [0], None, [256], [0, 256])

plt.subplot(1, 2, 1)
plt.imshow(bart, cmap='gray', vmin=0, vmax=255)
plt.title("Oryginal")
plt.axis('off')

plt.subplot(1, 2, 2)
plt.plot(hist)
plt.title("Histogram")
plt.show()

LOWER = 185 
UPPER = 216

mask_lower = (bart >= LOWER).astype(int)
mask_upper = (bart <= UPPER).astype(int)

binarized = (mask_lower * mask_upper * 255).astype(np.uint8)

plt.imshow(binarized, cmap='gray', vmin=0, vmax=255)
plt.title(f"Segmentacja ({LOWER} - {UPPER})")
plt.axis('off')
plt.show()

### Czyszczenie plików tymczasowych
Skrypt usuwający pliki konfiguracyjne oraz obrazy pobrane do testów lokalnych z poziomu notatnika.

In [ ]:
import os
import glob

trash_files = glob.glob('*.png*') + glob.glob('*.jpg*') + glob.glob('*.bmp*')

for file in trash_files:
    try:
        os.remove(file)
    except OSError:
        pass

print("Sprzątanie zakończone! Przestrzeń robocza oczyszczona z plików tymczasowych.")